In [1]:
import pandas as pd
pd.options.plotting.backend = "plotly"

from summer2 import CompartmentalModel
from summer2.parameters import Parameter

Let's try to update the previous model from 03_seir_births_deaths with a simple adjustment: We maintain an average dwell time in the latent/exposed compartment of 1 year, but instead of a progression rate of 1 (implying that 100% of infected people progress to the infectious/disease compartment if they do not die before that), we add a competing risk to the system: 90% of infected people clear their infection, ie. they bypass the infectious compartment altogether and enter directly into the recovered compartment. This leaves 10% of people to progress to disease:

In [2]:
# Same function as in notebook 02_seir, but with birth and death flows
def build_seir_model(
        config: dict,
) -> CompartmentalModel:
    
    """ 
    Args: 
        config: dict with configurations other than parameters
    Returns:
        model: the summer model object
    """
    compartments = (
        "susceptible",
        "exposed",
        "infectious",
        "recovered",
    )
    analysis_times = (config["start_time"], config["end_time"])

    model = CompartmentalModel(
        times = analysis_times,
        compartments = compartments,
        infectious_compartments = ("infectious",),
    )

    model.set_initial_population(
        distribution={
            "susceptible": config["population"] - config["seed"],
            "infectious": config["seed"], 
        },
    )

    # Demographics
    model.add_crude_birth_flow(
        "births",
        Parameter("birth_rate"),
        "susceptible",
    )

    model.add_universal_death_flows(
        "background_mortality",
        death_rate=Parameter("death_rate",)
    )

    # Disease flows
    model.add_infection_frequency_flow(
        name="infection",
        contact_rate=Parameter("contact_rate"),
        source="susceptible",
        dest="exposed",
    )
    model.add_transition_flow(
        name="progression",
        fractional_rate=Parameter("progression"),
        source="exposed",
        dest="infectious",
    )
    model.add_transition_flow(
        name="recovery",
        fractional_rate=Parameter("recovery"),
        source="infectious",
        dest="recovered",
    )

    # Now we add a clearance flow
    model.add_transition_flow(
        name="natural_clearance",
        fractional_rate=Parameter("clearance_rate"),
        source="exposed",
        dest="recovered"
    )

    return model

In [3]:
# We use the same configurations:
model_config = {
    "population": 48928.0, # TB study pop in 2004 in Guinea-Bissau
    "seed": 144.0, # Number of cases diagnosed in 2004
    "start_time": 2004,
    "end_time": 2020,
}

# Create instance of SEIR model with these configs
seir_model = build_seir_model(model_config)

In [6]:
# But we add the clearance rate to the parameters (and adjust the progression rate accordingly)
parameters = {
    "recovery": 0.15,
    "contact_rate": 10.0,
    "death_rate": 0.022, # Average life expectancy was 45.5 in 2004
    "birth_rate": 0.022, # We start by matching the death rate to keep the population stable over time

    "clearance_rate": 0.8, # Say 80% of infected peolpe actually recover naturally
    "progression": 0.2, # Should then be 20% to give the average dwell time in the latent compartment of 1 year
}

In [7]:
# Run and plot:
seir_model.run(parameters=parameters)
compartment_values = seir_model.get_outputs_df()
compartment_values.plot(
    labels={"index": "time", "value": "compartment size"}
)

The model still assumes introduction of the pathogen into a completely susceptible population, and we still do not account for vaccines and treatment. We also assume that recovered people do not become susceptible again, which is why the susceptible compartment is depleted over time, while the recovered compartment grows in size. 

There are multiple factors we have not yet accounted for:
- Waning immunity
- Treamtent and vaccine coverage
- Differences in disease dynamics by sex, age, and HIV